In [1]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [2]:
RANDOM_STATE = 12345

# Load data

In [3]:
housing_data = fetch_california_housing(as_frame=True)
housing_df = housing_data.frame
feature_cols = list(housing_data.feature_names)
target_cols = list(housing_data.target_names)
housing_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [4]:
X = housing_df[feature_cols].values
y = housing_df[target_cols].squeeze()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_valid, X_test, y_valid, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=RANDOM_STATE)

print('Train', X_train.shape, y_train.shape)
print('Valid', X_valid.shape, y_valid.shape)
print('Test', X_test.shape, y_test.shape)

Train (16512, 8) (16512,)
Valid (2064, 8) (2064,)
Test (2064, 8) (2064,)


# LGBMRegressor

In [5]:
train_data = lgb.Dataset(X_train, label=y_train, feature_name=feature_cols)
valid_data = lgb.Dataset(X_valid, label=y_valid, feature_name=feature_cols)
test_data = lgb.Dataset(X_test, label=y_test, feature_name=feature_cols)

In [6]:
params = {
    'boosting_type': 'gbdt',       # Gradient Boosting Decision Tree
    'objective': 'regression',     # L2 loss (Mean Squared Error); raw model output IS the prediction
    'n_estimators': 3,             # Number of boosting rounds
    'num_leaves': 4,               # Max tree leaves for base learners
}

regressor = lgb.train(train_set=train_data, params=params)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000254 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 16512, number of used features: 8
[LightGBM] [Info] Start training from score 2.064984


In [7]:
def print_tree(tree):
    for i in tree:
        if not math.isnan(float(i['split_gain'])):
            message = f"[{i['split_feature']}{i['decision_type']}{i['threshold']}], gain={i['split_gain']}"
        else:
            message = f"leaf={i['value']}"
        tab = '    '*(i['node_depth']-1)
        print(f"{tab}{message}")

In [8]:
# Convert all trees to a single DataFrame
tree_df = regressor.trees_to_dataframe()

In [9]:
# View split details for the first tree
tree = tree_df[tree_df['tree_index'] == 0].to_dict(orient='records')
print_tree(tree)

[MedInc<=5.075000000000001], gain=6791.72021484375
    [MedInc<=3.1261000000000005], gain=1729.4000244140625
        leaf=1.9961616050146855
        leaf=2.0687547311470325
    [MedInc<=6.589600000000001], gain=1250.6400146484375
        leaf=2.145682001705269
        leaf=2.2723055923787117


In [10]:
# View split details for the second tree
tree = tree_df[tree_df['tree_index'] == 1].to_dict(orient='records')
print_tree(tree)

[MedInc<=4.566450000000001], gain=5518.0
    [MedInc<=2.8367500000000008], gain=1068.6500244140625
        leaf=-0.06915779507132322
        leaf=-0.008945715219597729
    [MedInc<=6.369950000000001], gain=1469.989990234375
        leaf=0.055649818274405606
        leaf=0.17853293235442957


In [11]:
# View split details for the third tree
tree = tree_df[tree_df['tree_index'] == 2].to_dict(orient='records')
print_tree(tree)

[MedInc<=5.578600000000001], gain=4523.43017578125
    [MedInc<=3.5011000000000005], gain=1512.2900390625
        leaf=-0.04973568762197114
        [AveOccup<=2.35005995203837], gain=1007.0
            leaf=0.098023643395097
            leaf=-0.004072517368106745
    leaf=0.12719702393558371


# Algorithm

LightGBM is a gradient-boosted decision tree (GBDT). Like XGBoost it builds an additive model,

$$F_t(x) = F_{t-1}(x) + \eta\, f_t(x)$$

where $f_t$ is the $t$-th tree, $\eta$ is the learning rate.  

## Gradient and Hessian for squared error loss

For target $y_i$ and prediction $\hat y_i = F(x_i)$, with $l = \tfrac12(\hat y_i - y_i)^2$:

$$g_i = \frac{\partial l}{\partial \hat y_i} = \hat y_i - y_i \qquad h_i = \frac{\partial^2 l}{\partial \hat y_i^2} = 1$$

The **pseudo-residual** is the negative gradient, $r_i = y_i - \hat y_i = -g_i$, and the Hessian is
the constant $1$ for every sample.

## Baseline

The optimal constant prediction minimising $\sum \tfrac12(y_i - c)^2$ is the mean of the target:

$$F_0 = \bar y = \frac{1}{n}\sum_{i=1}^{n} y_i$$

## Split gain and leaf weight

With aggregated gradient $G=\sum g_i$ and Hessian $H=\sum h_i$ over a node (using $\lambda=$ `reg_lambda`,
default $0$), the per-node similarity score and the optimal leaf weight are

$$S = \frac{G^2}{H+\lambda}, \qquad w^{*} = -\frac{G}{H+\lambda} = \frac{\sum_{i}(y_i-\hat y_i)}{n+\lambda}$$

(since $G=-\sum r_i$ and $H=n$, $w^*$ is just the mean residual for $\lambda=0$).  
A candidate split is scored by

$$\text{gain} = S_L + S_R - S_{\text{parent}}$$

## Leaf-wise growth
Repeatedly splits the leaf with the largest `gain`.
The prediction update applies the shrunk leaf weight:

$$F_t(x) = F_{t-1}(x) + \eta\, w^{*}$$

> **Note on the `value` column.** `trees_to_dataframe()` reports each node's `value` as the score a
> sample would have if it stopped at that node. The first tree folds in the baseline, so its leaves
> show $F_0 + \eta\, w^{*}$; from the second tree on the baseline is $0$, so their leaves show just
> $\eta\, w^{*}$.

In [12]:
def greedy_find_bin(distinct_values, counts, num_distinct_values, max_bin, total_cnt, min_data_in_bin=3):
    """ Returns the list of bin upper edges (candidate split thresholds). When there are fewer
    distinct values than bins it just places a midpoint every `min_data_in_bin` samples; otherwise
    it greedily merges values into ~equal-frequency bins. `mean_bin_size` is float, 
    so we deliberately use float division to reproduce the exact edges."""
    
    bin_upper_bound = []
    if num_distinct_values <= max_bin:
        cur_cnt_inbin = 0
        for i in range(num_distinct_values - 1):
            cur_cnt_inbin += counts[i]
            if cur_cnt_inbin >= min_data_in_bin:
                val = (distinct_values[i] + distinct_values[i + 1]) / 2.0
                bin_upper_bound.append(val)
                cur_cnt_inbin = 0
    else:
        if min_data_in_bin > 0:
            max_bin = min(max_bin, int(total_cnt / min_data_in_bin))
            max_bin = max(max_bin, 1)
            
        mean_bin_size = total_cnt / max_bin
        rest_bin_cnt = max_bin
        rest_sample_cnt = total_cnt
        is_big_count_value = [False] * num_distinct_values
        
        for i in range(num_distinct_values):
            if counts[i] >= mean_bin_size:
                is_big_count_value[i] = True
                rest_bin_cnt -= 1
                rest_sample_cnt -= counts[i]
                
        mean_bin_size = rest_sample_cnt / rest_bin_cnt
        upper_bounds = [np.inf] * max_bin
        lower_bounds = [np.inf] * max_bin
        bin_cnt = 0
        lower_bounds[bin_cnt] = distinct_values[0]
        cur_cnt_inbin = 0
        
        for i in range(num_distinct_values - 1):
            if not is_big_count_value[i]:
                rest_sample_cnt -= counts[i]
            cur_cnt_inbin += counts[i]
            cond = is_big_count_value[i + 1] & (cur_cnt_inbin >= max(1.0, mean_bin_size * 0.5))
            if is_big_count_value[i] | (cur_cnt_inbin >= mean_bin_size) | cond:
                upper_bounds[bin_cnt] = distinct_values[i]
                bin_cnt += 1
                lower_bounds[bin_cnt] = distinct_values[i + 1]
                if bin_cnt >= max_bin - 1:
                    break
                cur_cnt_inbin = 0
                if not is_big_count_value[i]:
                    rest_bin_cnt -= 1
                    mean_bin_size = rest_sample_cnt / rest_bin_cnt
                    
        bin_cnt += 1
        for i in range(bin_cnt - 1):
            val = float((upper_bounds[i] + lower_bounds[i + 1]) / 2.0)
            if not bin_upper_bound or val > bin_upper_bound[-1]:
                bin_upper_bound.append(val)
                
    return bin_upper_bound

def find_bin(Xf, max_bin=255, min_data_in_bin=3, KZERO=1e-35):
    """Split the value range around zero, reserve one bin as the zero boundary, 
    and distribute the remaining bins between the negative and positive sides 
    proportionally to their sample counts."""

    # sort the column and count how many training samples take each value.
    distinct_values, counts = np.unique(Xf, return_counts=True)
    num_distinct_values = len(distinct_values)
    total_sample_cnt = len(Xf)

    # split the value range around zero into negetive (left), zero (center) and positive (right)
    left_cnt_data = int(np.sum(counts[distinct_values <= -KZERO]))
    cnt_zero = int(np.sum(counts[(distinct_values > -KZERO) & (distinct_values <= KZERO)]))
    right_cnt_data = int(np.sum(counts[distinct_values > KZERO]))

    left_cnt = -1
    for i in range(num_distinct_values):
        if distinct_values[i] > -KZERO:
            left_cnt = i
            break
            
    if left_cnt < 0:
        left_cnt = num_distinct_values

    bounds = []
    if left_cnt > 0 and max_bin > 1:
        left_max_bin = max(1, int(left_cnt_data / (total_sample_cnt - cnt_zero) * (max_bin - 1)))
        left_bounds = greedy_find_bin(distinct_values, counts, left_cnt, left_max_bin, left_cnt_data, min_data_in_bin)
        if len(left_bounds) > 0:
            left_bounds[-1] = -KZERO
        bounds += left_bounds

    right_start = -1
    for i in range(left_cnt, num_distinct_values):
        if distinct_values[i] > KZERO:
            right_start = i
            break

    right_max_bin = max_bin - 1 - len(bounds)
    if right_start >= 0 and right_max_bin > 0:
        right_bounds = greedy_find_bin(distinct_values[right_start:], counts[right_start:],
                                       num_distinct_values - right_start, right_max_bin,
                                       right_cnt_data, min_data_in_bin)
        bounds.append(KZERO)
        bounds += right_bounds
    else:
        bounds.append(np.inf)
    return bounds

In [13]:
def similarity_score(residuals, hessians, reg_lambda=0):
    """S = G^2 / (H + lambda).  residual = y - y_hat = -g, so G^2 is the same either sign."""
    G = np.sum(residuals)
    H = np.sum(hessians)
    return (G * G) / (H + reg_lambda)

def leaf_value(residuals, hessians, reg_lambda=0, learning_rate=0.1):
    """Shrunk optimal leaf weight:  eta * w* = eta * sum(y - y_hat) / (n + lambda).
    For L2 loss this is just eta * mean(residual) when lambda = 0."""
    G = np.sum(residuals)
    H = np.sum(hessians)
    return learning_rate * G / (H + reg_lambda)

In [14]:
def find_best_split(X, residuals, hessians, feature_cols, possible_thresholds, 
                    reg_lambda=0.0, min_data_in_leaf=20, min_sum_hessian_in_leaf=1e-3):
    
    similarity_parent = similarity_score(residuals, hessians, reg_lambda)
    
    max_gain = -1
    best = None
    for idx in range(len(feature_cols)):
        Xf = X[:, idx]
        for thres in possible_thresholds[idx]:
            # split the node into a "left" (<=) group and a "right" (>) group
            mask_left = Xf <= thres
            mask_right = ~mask_left
            n_left = int(mask_left.sum())
            n_right = int(mask_right.sum())
            if n_left == 0 or n_right == 0:
                continue
            if n_left < min_data_in_leaf or n_right < min_data_in_leaf:
                continue

            r_left, h_left = residuals[mask_left], hessians[mask_left]
            r_right, h_right = residuals[mask_right], hessians[mask_right]
            H_left = np.sum(h_left)
            H_right = np.sum(h_right)
            if H_left < min_sum_hessian_in_leaf or H_right < min_sum_hessian_in_leaf:
                continue

            sim_left = similarity_score(r_left, h_left, reg_lambda)
            sim_right = similarity_score(r_right, h_right, reg_lambda)
            gain = sim_left + sim_right - similarity_parent

            if (gain > max_gain) and (gain > 0):
                max_gain = gain
                best = {
                    'feature': feature_cols[idx], 'feature_idx': idx,
                    'threshold': thres, 'gain': gain,
                    'left':  (X[mask_left],  r_left,  h_left),
                    'right': (X[mask_right], r_right, h_right),
                }
    return best

def find_next_split_node(leaf_nodes, feature_cols, possible_thresholds):
    """Leaf-wise (best-first) growth: pick the existing leaf with the largest split gain."""
    best_idx = None
    best_split = None
    best_gain = -1
    for idx, node in enumerate(leaf_nodes):
        Xn, rn, hn = node[1], node[2], node[3]
        split = find_best_split(Xn, rn, hn, feature_cols, possible_thresholds)
        if split is not None and split['gain'] > best_gain:
            best_gain = split['gain']
            best_idx = idx
            best_split = split
    return best_idx, best_split

In [15]:
def build_tree(X, residuals, hessians, feature_cols, possible_thresholds, num_leaves=4, init_score=0.0):
    """Grow a tree leaf-wise (best-first) until num_leaves is reached.

    Returns the root node. Each node is a dict with feature/threshold/gain/
    left/right/depth; leaves additionally carry a `value`."""
    
    root = {'feature': None, 'threshold': None, 'gain': None, 'left': None, 'right': None, 'depth': 0}
    
    # each leaf entry is (node_dict, X_subset, residuals_subset, hessians_subset)
    leaves = [(root, X, residuals, hessians)]
    while len(leaves) < num_leaves:
        best_idx, best_split = find_next_split_node(leaves, feature_cols, possible_thresholds)
        if best_split is None:
            break
        node, _, _, _ = leaves.pop(best_idx)
        node['feature'] = best_split['feature']
        node['threshold'] = best_split['threshold']
        node['gain'] = best_split['gain']
        d = node['depth']
        left_child = {'feature': None, 'threshold': None, 'gain': None, 'left': None, 'right': None, 'depth': d + 1}
        right_child = {'feature': None, 'threshold': None, 'gain': None, 'left': None, 'right': None, 'depth': d + 1}
        node['left'] = left_child
        node['right'] = right_child
        leaves.insert(best_idx, (right_child, best_split['right'][0], best_split['right'][1], best_split['right'][2]))
        leaves.insert(best_idx, (left_child, best_split['left'][0], best_split['left'][1], best_split['left'][2]))

    for (node, _, rn, hn) in leaves:
        node['value'] = init_score + leaf_value(rn, hn)

    return root

def print_scratch_tree(node, indent=0):
    pad = '    ' * indent
    if node['left'] is None:
        print(f"{pad}leaf={node['value']}")
    else:
        print(f"{pad}[{node['feature']}<={node['threshold']:.8f}], gain={node['gain']}")
        print_scratch_tree(node['left'], indent + 1)
        print_scratch_tree(node['right'], indent + 1)

## Build candidate thresholds of each feature.

LightGBM never scans every distinct value when searching for a split. Instead, *once up front*,
it reduces each feature to a small set of **histogram bins**, and the split search later only ever tests the **bin edges** as candidate thresholds. 

`find_bin()` turns one feature column `Xf` into that list of candidate thresholds, which we store in `possible_thresholds`.

1. sort the column and count how many training samples take each value.
2. split the value range around zero into negetive (left), zero (center) and positive (right)
3. compute the number of bins for negative and positive side. They are divided proportionally to sample counts
4. bin the negative side and force the last edge to -1e-35 so the negative region ends exactly at the zero boundary.
5. bin the zero side at 1e-35
6. bin the positve side

`greedy_find_bin()` used to find the candidate thresholds for both negative and positive side.
- **Few distinct values (`num_distinct_values <= max_bin`):**  
    walk the sorted values and drop a
    midpoint `(v[i] + v[i+1])/2` every time a running count reaches `min_data_in_bin` (default=3).
  
- **Many distinct values:**  
    greedily merge consecutive values into roughly **equal-frequency** bins. The target fill is `mean_bin_size = total_cnt / max_bin`.  
    Any single value whose own count already exceeds it (`is_big_count_value`) gets its own bin.  
    A new edge is opened once the running count reaches `mean_bin_size`.  
    The edge value is the midpoint of the two boundary distinct values, `(upper_bounds[i] + lower_bounds[i+1]) / 2`.

In [16]:
possible_thresholds = dict()
for idx in range(len(feature_cols)):
    possible_thresholds[idx] = find_bin(X_train[:, idx], max_bin=255)

## Tree 0 — initialize the baseline

Start from the constant prediction $F_0 = \bar y$ and form the first pseudo-residuals
$r_i = y_i - F_0$ and Hessians $h_i = 1$.

In [17]:
baseline = y_train.mean()
F_prev = np.full(len(y_train), baseline)   # F0: constant initial prediction

residuals = y_train - F_prev
hessians = np.ones(len(y_train))

## Fit tree 0

Grow leaf-wise (always split the leaf with the largest gain). The first tree folds the baseline
into its leaf `value`s, so each leaf shows $F_0 + \eta\, w^{*}$ — matching LightGBM's
`trees_to_dataframe()` output for `tree_index == 0` above.

In [18]:
root = build_tree(X_train, residuals, hessians, feature_cols, possible_thresholds, num_leaves=4, init_score=baseline)
print_scratch_tree(root)

[MedInc<=5.07500000], gain=6791.717856115082
    [MedInc<=3.12610000], gain=1729.4018691106933
        leaf=1.9961616056215006
        leaf=2.068754731658688
    [MedInc<=6.58960000], gain=1250.6411465469982
        leaf=2.1456820016088356
        leaf=2.272305589189516


## Update predictions

Add the first tree's contribution to the running prediction. For regression there is no link
function, so `predict(...)` returns the raw scores $F$ directly — we use it as $F_1$ and recompute
residuals for the next tree.

In [19]:
F_prev = regressor.predict(X_train, num_iteration=1)   # F1 (predictions == raw scores)

## Tree 1

Recompute residuals $y - F_1$ and Hessians ($1$), then grow the second tree. From the second tree on
the baseline is $0$, so leaves show just $\eta\, w^{*}$.

In [20]:
residuals = y_train - F_prev
hessians = np.ones(len(y_train))

root = build_tree(X_train, residuals, hessians, feature_cols, possible_thresholds, num_leaves=4, init_score=0.0)
print_scratch_tree(root)

[MedInc<=4.56645000], gain=5518.00354582377
    [MedInc<=2.83675000], gain=1068.6493679600242
        leaf=-0.06915779551741902
        leaf=-0.008945715948900111
    [MedInc<=6.36995000], gain=1469.9856543913984
        leaf=0.05564981801506181
        leaf=0.1785329277159976


In [21]:
F_prev = regressor.predict(X_train, num_iteration=2)   # F2 (predictions == raw scores)

## Tree 2

One more boosting round, using $F_2$ as the current prediction. Notice the tree grows unbalanced —
leaf-wise growth splits the left region twice before ever touching the right region, because that
is where the gain is largest.

In [22]:
residuals = y_train - F_prev
hessians = np.ones(len(y_train))

root = build_tree(X_train, residuals, hessians, feature_cols, possible_thresholds, num_leaves=4, init_score=0.0)
print_scratch_tree(root)

[MedInc<=5.57860000], gain=4523.428325423808
    [MedInc<=3.50110000], gain=1512.285152920469
        leaf=-0.04973568731157991
        [AveOccup<=2.35005995], gain=1007.0002127311054
            leaf=0.09802364259808777
            leaf=-0.0040725171866467305
    leaf=0.12719702532235372
